In [2]:
# Libraries
import pandas as pd
from sqlalchemy import create_engine, text
from pathlib import Path

# Current working directory
print('Arbeitsverzeichnis:', Path.cwd())

#NICHTS ANPASSEN: Imports und Path.cwd()
#ERKENNTNIS: Relative Dateipfade beziehen sich auf das Arbeitsverzeichnis des Notebooks

Arbeitsverzeichnis: /workspaces/DS_HS2026_LNW/DS_HS2026_LNW_I_Examples/AP01/AP01 Mock Exam


In [3]:
# Set up database connection
user = 'pgadmin'                                                                #Benutzername für PostgreSQL
password = 'geheim'                                                             #Passwort des Datenbankbenutzers
host = 'localhost'                                                              #Adresse des PostgreSQL-Servers
port = '5432'                                                                   #Port, über den PostgreSQL erreichbar ist
database = 'postgres'                                                           #Name der Datenbank, mit der verbunden wird

# Create Connection URL
db_connection_url = (
    f'postgresql://{user}:{password}@{host}:{port}/{database}'                  #Verbindungsadresse aus den oben definierten Angaben zusammensetzen
)                                                                               

# Create SQLAlchemy Engine
engine = create_engine(db_connection_url)                                       #Engine erstellen; Verbindung wird erst bei Bedarf geöffnet

# Test database connection
with engine.connect() as connection:                                            #Verbindung öffnen und nach dem Block automatisch schliessen
    result = connection.execute(text('SELECT current_database()'))              #Namen der aktuellen Datenbank abfragen
    print('Verbundene Datenbank:', result.scalar())                             #Einzelnen Ergebniswert auslesen und anzeigen

# Dispose the engine
engine.dispose()                                                                #Verbindungen im Pool schliessen und Ressourcen freigeben

Verbundene Datenbank: postgres


### Retrieve all rows and all columns from the bicycles table

In [4]:
# Create SQLAlchemy Engine
engine = create_engine(db_connection_url)                                       #Engine für die Datenbankverbindung erstellen

# Execute SQL query
df_bicycles = pd.read_sql_query("""
    SELECT *                                                --Alle Spalten auswählen
    FROM bicycles;                                          --Daten aus der Tabelle bicycles abrufen
""", con=engine)                                                                #SQL ausführen und Ergebnis als DataFrame speichern

# Dispose the engine
engine.dispose()                                                                #Verbindungen im Pool schliessen und Ressourcen freigeben

# Show results
display(df_bicycles)                                                            #Ergebnistabelle anzeigen

,bicycleid,make,model,year,framenr,price,status
0,C7ZGWFLBUV,Trek,Domane,2019,A15 0LX,5066.94,In Maintenance
1,C875HZ1AG6,Merida,Reacto,2021,435N,5529.38,Sold
2,CDONEWSWFP,Merida,Big Nine,2021,QBA-572,1518.33,In Maintenance
3,C2UQZP803C,Trek,Marlin,2023,5-92778W,5038.99,In Maintenance
4,C2WFIK6WEP,Merida,Scultura,2020,300 VLT,1548.19,Sold
...,...,...,...,...,...,...,...
1995,CLH919EIKV,Trek,Emonda,2020,911 LYI,2376.02,Available
1996,CP3F48BE9Y,Merida,Scultura,2022,YPU 135,2833.36,Available
1997,CHBXQ3SSNU,Merida,Scultura,2019,059-GNM,1629.91,In Maintenance
1998,CP5CTME2ZD,Trek,Marlin,2021,02V 6510,6176.55,In Maintenance


# JOINS

### Retrieve each sale's ID, sale price and the first and last name of the salesperson who made the sale

In [5]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT s.saleid,                                             --Verkaufs-ID aus der Tabelle sales auswählen
           s.saleprice,                                          --Verkaufspreis auswählen
           sp.firstname,                                         --Vornamen des zugehörigen Verkäufers auswählen (Tabelle SalesPersons)
           sp.lastname                                           --Nachnamen des zugehörigen Verkäufers auswählen
    FROM sales AS s                                              --Tabelle sales verwenden; s ist ihr Kurzname
    INNER JOIN salespersons AS sp                                --Verkäufertabelle ergänzen; sp ist ihr Kurzname
        ON s.salespersonid = sp.salespersonid;                   --Über dieselbe Verkäufer-ID verbinden; nur passende Kombinationen behalten
""", con=engine)

engine.dispose()
display(df_spickap01)

,saleid,saleprice,firstname,lastname
0,SLBF6MT8J2,3480,Robert,Lyons
1,SQX72758TF,2680,Heather,Brown
2,S5FDZ50KNW,3000,Michael,Zhang
3,S79LBQUU41,4320,Lauren,Robinson
4,SIECZIGLOC,1900,Karen,Randall
...,...,...,...,...
679,STJ906IYGY,3120,Michael,Zhang
680,S1L1CI9G7K,1550,Heather,Brown
681,SJQ2VP6CY4,3360,Karen,Randall
682,SJ3XI5MJ1Q,1630,Brian,Klein


### Determine the number of bicycles sold and total sales revenue for each salesperson who has made a sale. Display their first name, last name, phone, email, total_bicycles_sold and total_sales_amount. Round the revenue to two decimal places and sort it in descending order

In [6]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT sp.firstname,                                                        --Vornamen des Verkäufers anzeigen
           sp.lastname,                                                         --Nachnamen des Verkäufers anzeigen
           sp.phone,                                                            --Telefonnummer anzeigen
           sp.email,                                                            --E-Mail-Adresse anzeigen
           COUNT(s.saleid) AS total_bicycles_sold,                              --Verkäufe pro Verkäufer zählen
           ROUND(CAST(SUM(s.saleprice) AS NUMERIC), 2) AS total_sales_amount    --Umsatz pro Verkäufer addieren und auf zwei Stellen runden
    FROM salespersons AS sp                                                     --Verkäufertabelle verwenden; sp ist ihr Kurzname
    INNER JOIN sales AS s                                                       --Verkäufe ergänzen; nur Verkäufer mit passenden Verkäufen behalten
        ON sp.salespersonid = s.salespersonid                                   --Tabellen über die Verkäufer-ID verbinden
    GROUP BY sp.salespersonid, sp.firstname, sp.lastname, sp.phone, sp.email    --Nach Verkäufer-ID und allen nicht aggregierten Ausgabespalten gruppieren
    ORDER BY total_sales_amount DESC;                                           --Verkäufer mit dem höchsten Gesamtumsatz zuerst anzeigen
""", con=engine)

engine.dispose()
display(df_spickap01)

,firstname,lastname,phone,email,total_bicycles_sold,total_sales_amount
0,Heather,Brown,+41 (0)46 288 75 19,heather.brown@example.com,82,267840.0
1,Christine,Harris,+41 (0)61 523 86 58,christine.harris@example.com,74,237420.0
2,Karen,Randall,+41 (0)66 616 17 40,karen.randall@example.com,75,232950.0
3,Robert,Lyons,+41 (0)40 422 23 33,robert.lyons@example.com,69,232330.0
4,Lauren,Robinson,+41 (0)71 214 12 13,lauren.robinson@example.com,69,216840.0
5,Ryan,Baxter,+41 (0)40 706 53 70,ryan.baxter@example.com,72,208490.0
6,Nicole,Carter,+41 (0)77 266 76 74,nicole.carter@example.com,66,207940.0
7,Cathy,Becker,+41 (0)23 263 98 44,cathy.becker@example.com,67,200340.0
8,Michael,Zhang,+41 (0)63 311 51 92,michael.zhang@example.com,59,180090.0
9,Brian,Klein,+41 (0)77 449 26 62,brian.klein@example.com,51,174610.0


### Determine the number of sales and total sales revenue for every salesperson, including those with no sales. Display salespersonid, firstname, lastname, sales_count and total_sales_amount. Show zero for salespeople with no sales, round revenue to two decimal places and sort by revenue in descending order

In [7]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT sp.salespersonid,                                                                --Verkäufer-ID anzeigen
           sp.firstname,                                                                    --Vornamen anzeigen
           sp.lastname,                                                                     --Nachnamen anzeigen
           COUNT(s.saleid) AS sales_count,                                                  --Nur vorhandene Verkäufe zählen; ohne Verkauf ergibt sich 0
           ROUND(CAST(COALESCE(SUM(s.saleprice), 0) AS NUMERIC), 2) AS total_sales_amount   --Umsatz summieren; fehlende Summe durch 0 ersetzen und runden
    FROM salespersons AS sp                                                                 --Von der Tabelle aller Verkäufer ausgehen
    LEFT JOIN sales AS s                                                                    --Alle Verkäufer behalten, auch ohne passenden Verkauf
        ON sp.salespersonid = s.salespersonid                                               --Verkäufe über die Verkäufer-ID zuordnen
    GROUP BY sp.salespersonid, sp.firstname, sp.lastname                                    --Eine Gruppe pro Verkäufer bilden
    ORDER BY total_sales_amount DESC;                                                       --Höchsten Gesamtumsatz zuerst anzeigen
""", con=engine)

engine.dispose()
display(df_spickap01)

,salespersonid,firstname,lastname,sales_count,total_sales_amount
0,SPCZZUOCF9,Heather,Brown,82,267840.0
1,SPGK07EO84,Christine,Harris,74,237420.0
2,SPOHEVXP0T,Karen,Randall,75,232950.0
3,SP0LU6X4G6,Robert,Lyons,69,232330.0
4,SPCGP1HZPG,Lauren,Robinson,69,216840.0
5,SPLO7OLQSK,Ryan,Baxter,72,208490.0
6,SPL5PTPR1D,Nicole,Carter,66,207940.0
7,SPOELVJOXX,Cathy,Becker,67,200340.0
8,SP86XCRI7B,Michael,Zhang,59,180090.0
9,SPXRVM1B8C,Brian,Klein,51,174610.0


### Retrieve the ID, first name and last name of all salespeople who have never made a sale

In [8]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT sp.salespersonid,                            --Verkäufer-ID auswählen
           sp.firstname,                                --Vornamen auswählen
           sp.lastname                                  --Nachnamen auswählen
    FROM salespersons AS sp                             --Von der Tabelle aller Verkäufer ausgehen
    LEFT JOIN sales AS s                                --Alle Verkäufer behalten, auch ohne passenden Verkauf
        ON sp.salespersonid = s.salespersonid           --Verkäufe über die Verkäufer-ID zuordnen
    WHERE s.saleid IS NULL;                             --Nur Verkäufer ohne passenden Verkauf behalten
""", con=engine)

engine.dispose()
display(df_spickap01)

,salespersonid,firstname,lastname


### Retrieve every salesperson's ID, first name and last name, together with their sale IDs and sale prices. Include salespeople with no sales

In [9]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT sp.salespersonid,                            --Verkäufer-ID auswählen
           sp.firstname,                                --Vornamen auswählen
           sp.lastname,                                 --Nachnamen auswählen
           s.saleid,                                    --Zugehörige Verkaufs-ID anzeigen
           s.saleprice                                  --Zugehörigen Verkaufspreis anzeigen
    FROM sales AS s                                     --Verkäufe stehen auf der linken Seite
    RIGHT JOIN salespersons AS sp                       --Alle Verkäufer der rechten Tabelle behalten
        ON s.salespersonid = sp.salespersonid;          --Verkäufe zuordnen; ohne Verkauf bleiben die Verkaufsspalten NULL
""", con=engine)

engine.dispose()
display(df_spickap01)

,salespersonid,firstname,lastname,saleid,saleprice
0,SP0LU6X4G6,Robert,Lyons,SLBF6MT8J2,3480
1,SPCZZUOCF9,Heather,Brown,SQX72758TF,2680
2,SP86XCRI7B,Michael,Zhang,S5FDZ50KNW,3000
3,SPCGP1HZPG,Lauren,Robinson,S79LBQUU41,4320
4,SPOHEVXP0T,Karen,Randall,SIECZIGLOC,1900
...,...,...,...,...,...
679,SP86XCRI7B,Michael,Zhang,STJ906IYGY,3120
680,SPCZZUOCF9,Heather,Brown,S1L1CI9G7K,1550
681,SPOHEVXP0T,Karen,Randall,SJQ2VP6CY4,3360
682,SPXRVM1B8C,Brian,Klein,SJ3XI5MJ1Q,1630


### Retrieve all sales and all salespeople, including unmatched records from either table. Display the sale ID, sale price, salesperson ID from each table, first name and last name

In [10]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT s.saleid,                                    --Verkaufs-ID auswählen
           s.saleprice,                                 --Verkaufspreis auswählen
           s.salespersonid AS sale_salespersonid,       --Im Verkauf gespeicherte Verkäufer-ID anzeigen
           sp.salespersonid AS salesperson_id,          --Verkäufer-ID aus der Verkäufertabelle anzeigen
           sp.firstname,                                --Vornamen des Verkäufers auswählen
           sp.lastname                                  --Nachnamen des Verkäufers auswählen
    FROM sales AS s                                     --Verkaufstabelle auf der linken Seite verwenden
    FULL OUTER JOIN salespersons AS sp                  --Alle Datensätze beider Tabellen behalten
        ON s.salespersonid = sp.salespersonid;          --Passende Datensätze verbinden; fehlende Gegenseite mit NULL ergänzen
""", con=engine)

engine.dispose()
display(df_spickap01)

,saleid,saleprice,sale_salespersonid,salesperson_id,firstname,lastname
0,SLBF6MT8J2,3480,SP0LU6X4G6,SP0LU6X4G6,Robert,Lyons
1,SQX72758TF,2680,SPCZZUOCF9,SPCZZUOCF9,Heather,Brown
2,S5FDZ50KNW,3000,SP86XCRI7B,SP86XCRI7B,Michael,Zhang
3,S79LBQUU41,4320,SPCGP1HZPG,SPCGP1HZPG,Lauren,Robinson
4,SIECZIGLOC,1900,SPOHEVXP0T,SPOHEVXP0T,Karen,Randall
...,...,...,...,...,...,...
679,STJ906IYGY,3120,SP86XCRI7B,SP86XCRI7B,Michael,Zhang
680,S1L1CI9G7K,1550,SPCZZUOCF9,SPCZZUOCF9,Heather,Brown
681,SJQ2VP6CY4,3360,SPOHEVXP0T,SPOHEVXP0T,Karen,Randall
682,SJ3XI5MJ1Q,1630,SPXRVM1B8C,SPXRVM1B8C,Brian,Klein


### Retrieve sales without a matching salesperson and salespeople without any sales. Display the sale ID, sale price, salesperson ID from each table, first name and last name

In [11]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT s.saleid,                                    --Verkaufs-ID auswählen
           s.saleprice,                                 --Verkaufspreis auswählen
           s.salespersonid AS sale_salespersonid,       --Im Verkauf gespeicherte Verkäufer-ID anzeigen
           sp.salespersonid AS salesperson_id,          --Verkäufer-ID aus der Verkäufertabelle anzeigen
           sp.firstname,                                --Vornamen auswählen
           sp.lastname                                  --Nachnamen auswählen
    FROM sales AS s                                     --Verkaufstabelle auf der linken Seite verwenden
    FULL OUTER JOIN salespersons AS sp                  --Zunächst alle Datensätze beider Tabellen behalten
        ON s.salespersonid = sp.salespersonid           --Datensätze über die Verkäufer-ID verbinden
    WHERE s.saleid IS NULL                              --Verkäufer ohne passenden Verkauf auswählen
       OR sp.salespersonid IS NULL;                     --Zusätzlich Verkäufe ohne passenden Verkäufer auswählen
""", con=engine)

engine.dispose()
display(df_spickap01)

,saleid,saleprice,sale_salespersonid,salesperson_id,firstname,lastname


### Retrieve all possible combinations of mechanics and bicycles. Display mechanicid, firstname, lastname, bicycleid, make and model

In [12]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT m.mechanicid,                        --Mechaniker-ID auswählen
           m.firstname,                         --Vornamen des Mechanikers auswählen
           m.lastname,                          --Nachnamen des Mechanikers auswählen
           b.bicycleid,                         --Fahrrad-ID auswählen
           b.make,                              --Fahrradmarke auswählen
           b.model                              --Fahrradmodell auswählen
    FROM mechanics AS m                         --Mechanikertabelle verwenden
    CROSS JOIN bicycles AS b;                   --Jeden Mechaniker mit jedem Fahrrad kombinieren; keine ON-Bedingung
""", con=engine)

engine.dispose()
display(df_spickap01)

,mechanicid,firstname,lastname,bicycleid,make,model
0,MV26A9TGJL,Erin,Williamson,C7ZGWFLBUV,Trek,Domane
1,MTA6ED59YC,Melissa,Miller,C7ZGWFLBUV,Trek,Domane
2,MBCL24BW42,Bryan,White,C7ZGWFLBUV,Trek,Domane
3,M8DA1H43YN,Shannon,Larsen,C7ZGWFLBUV,Trek,Domane
4,M8XUQMDR0P,Courtney,Ford,C7ZGWFLBUV,Trek,Domane
...,...,...,...,...,...,...
19995,M06GWF4R58,Tiffany,Wilson,CNJYADLN8R,Bianchi,Aria
19996,MLNW721B7L,Bradley,Mckinney,CNJYADLN8R,Bianchi,Aria
19997,MA4MHCDLIX,Cassandra,Sanford,CNJYADLN8R,Bianchi,Aria
19998,MCNI3BOSV1,Lisa,Hall,CNJYADLN8R,Bianchi,Aria


### Retrieve Bianchi bicycles sold from March to December 2023, including their bicycle ID, make, model, sale price, sale date, customer names and salesperson names. Sort by sale date in ascending order and limit the result to 10 rows

In [13]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT b.bicycleid,                                         --Fahrrad-ID auswählen
           b.make,                                              --Fahrradmarke auswählen
           b.model,                                             --Fahrradmodell auswählen
           s.saleprice,                                         --Tatsächlichen Verkaufspreis auswählen
           s.saledate,                                          --Verkaufsdatum auswählen
           c.firstname AS customer_firstname,                   --Kundenvornamen eindeutig benennen
           c.lastname AS customer_lastname,                     --Kundennachnamen eindeutig benennen
           sp.firstname AS salesperson_firstname,               --Verkäufervornamen eindeutig benennen
           sp.lastname AS salesperson_lastname                  --Verkäufernachnamen eindeutig benennen
    FROM bicycles AS b                                          --Von der Fahrradtabelle ausgehen
    INNER JOIN sales AS s                                       --Verkaufsinformationen ergänzen
        ON b.bicycleid = s.bicycleid                            --Fahrrad und Verkauf über die Fahrrad-ID verbinden
    INNER JOIN customers AS c                                   --Kundeninformationen ergänzen
        ON s.customerid = c.customerid                          --Verkauf und Kunde über die Kunden-ID verbinden
    INNER JOIN salespersons AS sp                               --Verkäuferinformationen ergänzen
        ON s.salespersonid = sp.salespersonid                   --Verkauf und Verkäufer über die Verkäufer-ID verbinden
    WHERE b.make = 'Bianchi'                                    --Nur Fahrräder der Marke Bianchi berücksichtigen
      AND s.saledate >= '2023-03-01'                            --Verkäufe ab dem 1. März 2023 einschliessen
      AND s.saledate < '2024-01-01'                             --Bis Ende Dezember einschliessen, auch bei gespeicherten Uhrzeiten
    ORDER BY s.saledate ASC                                     --Älteste Verkäufe zuerst anzeigen
    LIMIT 10;                                                   --Ausgabe auf maximal zehn Zeilen begrenzen
""", con=engine)

engine.dispose()
display(df_spickap01)

,bicycleid,make,model,saleprice,saledate,customer_firstname,customer_lastname,salesperson_firstname,salesperson_lastname


### Determine the number of services, average service cost and most recent service date for each mechanic and bicycle make across all years. Display the mechanic's first and last name, make, services_count, average_service_costs and last_service_date. Round average costs to two decimal places and sort them in descending order

In [14]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT m.firstname AS mechanic_firstname,                                   --Vornamen des Mechanikers anzeigen
           m.lastname AS mechanic_lastname,                                     --Nachnamen des Mechanikers anzeigen
           b.make,                                                              --Fahrradmarke anzeigen
           COUNT(*) AS services_count,                                          --Services pro Mechaniker und Marke zählen
           ROUND(CAST(AVG(sr.cost) AS NUMERIC), 2) AS average_service_costs,    --Durchschnittliche Servicekosten auf zwei Stellen runden
           MAX(sr.servicedate) AS last_service_date                             --Neuestes Servicedatum innerhalb der Gruppe ermitteln
    FROM mechanics AS m                                                         --Von der Mechanikertabelle ausgehen
    INNER JOIN servicerecords AS sr                                             --Durchgeführte Services ergänzen
        ON m.mechanicid = sr.mechanicid                                         --Services über die Mechaniker-ID zuordnen
    INNER JOIN bicycles AS b                                                    --Informationen zu den gewarteten Fahrrädern ergänzen
        ON sr.bicycleid = b.bicycleid                                           --Services über die Fahrrad-ID zuordnen
    GROUP BY m.mechanicid, m.firstname, m.lastname, b.make                      --Nach Mechaniker und Marke gruppieren; keine Gruppierung nach Jahr
    ORDER BY average_service_costs DESC;                                        --Höchste durchschnittliche Servicekosten zuerst anzeigen
""", con=engine)

engine.dispose()
display(df_spickap01)

,mechanic_firstname,mechanic_lastname,make,services_count,average_service_costs,last_service_date
0,Shannon,Larsen,Scott,65,39.76,2024-08-07
1,Cynthia,Pittman,Cannondale,74,38.61,2024-07-28
2,Shannon,Larsen,Specialized,75,37.41,2024-08-20
3,Courtney,Ford,Santa Cruz,82,36.41,2024-08-04
4,Erin,Williamson,Specialized,69,36.14,2024-08-17
...,...,...,...,...,...,...
95,Cynthia,Pittman,Specialized,82,27.53,2024-08-19
96,Lisa,Hall,Bianchi,70,27.47,2024-08-26
97,Cynthia,Pittman,Bianchi,77,26.81,2024-07-20
98,Melissa,Miller,Bianchi,71,26.49,2024-08-26


### Determine the average cost per year for services described as "Gear shifting adjustment". Display year and average_service_costs, rounded to two decimal places and sorted by average_service_costs in descending order

In [ ]:
engine = create_engine(db_connection_url)

df_spickap01 = pd.read_sql_query("""
    SELECT CAST(EXTRACT(YEAR FROM servicedate) AS INTEGER) AS year,             --Jahr aus dem Servicedatum auslesen und als Ganzzahl ausgeben
           ROUND(CAST(AVG(cost) AS NUMERIC), 2) AS average_service_costs        --Durchschnittliche Servicekosten pro Jahr auf zwei Stellen runden
    FROM servicerecords                                                         --Daten aus der Tabelle servicerecords abrufen
    WHERE servicedescription = 'Gear shifting adjustment'                       --Nur Services mit genau dieser Beschreibung berücksichtigen
    GROUP BY CAST(EXTRACT(YEAR FROM servicedate) AS INTEGER)             --Alle passenden Services desselben Jahres zusammenfassen
    ORDER BY average_service_costs DESC;                                --Jahr mit den höchsten durchschnittlichen Kosten zuerst anzeigen
""", con=engine)

engine.dispose()
display(df_spickap01)